In [1]:
pip install Flask flask-cors bcrypt pyjwt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import sqlite3
import uuid
import jwt
import datetime
import bcrypt
import os

app = Flask(__name__)
CORS(app)

# Secret key for JWT
SECRET_KEY = "your-secret-key"  # In production, use a secure random key stored in environment variables

# Database setup
def get_db_connection():
    conn = sqlite3.connect('warasat.db')
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_db_connection()
    
    # Create users table
    conn.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        user_type TEXT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # Create chats table
    conn.execute('''
    CREATE TABLE IF NOT EXISTS chats (
        id TEXT PRIMARY KEY,
        user_id TEXT NOT NULL,
        ulema_id TEXT NOT NULL,
        inheritance_data TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES users (id),
        FOREIGN KEY (ulema_id) REFERENCES users (id)
    )
    ''')
    
    # Create messages table
    conn.execute('''
    CREATE TABLE IF NOT EXISTS messages (
        id TEXT PRIMARY KEY,
        chat_id TEXT NOT NULL,
        sender_id TEXT NOT NULL,
        sender_type TEXT NOT NULL,
        content TEXT NOT NULL,
        is_read BOOLEAN DEFAULT 0,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (chat_id) REFERENCES chats (id),
        FOREIGN KEY (sender_id) REFERENCES users (id)
    )
    ''')
    
    # Insert some sample ulemas if they don't exist
    sample_ulemas = [
        ('ulema1', 'Mufti Abdullah', 'abdullah@example.com', 'password123', 'ulema'),
        ('ulema2', 'Mufti Ismail', 'ismail@example.com', 'password123', 'ulema'),
        ('ulema3', 'Mufti Yusuf', 'yusuf@example.com', 'password123', 'ulema')
    ]
    
    for ulema in sample_ulemas:
        # Check if ulema already exists
        cursor = conn.execute('SELECT id FROM users WHERE id = ?', (ulema[0],))
        if cursor.fetchone() is None:
            # Hash the password
            hashed_password = bcrypt.hashpw(ulema[3].encode('utf-8'), bcrypt.gensalt())
            conn.execute(
                'INSERT INTO users (id, name, email, password, user_type) VALUES (?, ?, ?, ?, ?)',
                (ulema[0], ulema[1], ulema[2], hashed_password, ulema[4])
            )
    
    conn.commit()
    conn.close()

# Initialize database
init_db()

# Helper functions
def generate_token(user_id, user_type):
    payload = {
        'user_id': user_id,
        'user_type': user_type,
        'exp': datetime.datetime.utcnow() + datetime.timedelta(days=1)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm='HS256')

def verify_token(token):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

def get_user_from_token(request):
    auth_header = request.headers.get('Authorization')
    if not auth_header or not auth_header.startswith('Bearer '):
        return None
    
    token = auth_header.split(' ')[1]
    return verify_token(token)

# Authentication routes
@app.route('/auth/signup', methods=['POST'])
def signup():
    data = request.json
    name = data.get('name')
    email = data.get('email')
    password = data.get('password')
    
    if not name or not email or not password:
        return jsonify({'success': False, 'message': 'Missing required fields'}), 400
    
    conn = get_db_connection()
    
    # Check if email already exists
    cursor = conn.execute('SELECT id FROM users WHERE email = ?', (email,))
    if cursor.fetchone() is not None:
        conn.close()
        return jsonify({'success': False, 'message': 'Email already registered'}), 400
    
    # Create new user
    user_id = str(uuid.uuid4())
    hashed_password = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())
    
    try:
        conn.execute(
            'INSERT INTO users (id, name, email, password, user_type) VALUES (?, ?, ?, ?, ?)',
            (user_id, name, email, hashed_password, 'user')
        )
        conn.commit()
        conn.close()
        
        return jsonify({'success': True, 'message': 'User registered successfully'}), 201
    except Exception as e:
        conn.close()
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/auth/login', methods=['POST'])
def login():
    data = request.json
    email = data.get('email')
    password = data.get('password')
    user_type = data.get('user_type', 'user')
    
    if not email or not password:
        return jsonify({'success': False, 'message': 'Missing email or password'}), 400
    
    conn = get_db_connection()
    cursor = conn.execute(
        'SELECT id, name, email, password, user_type FROM users WHERE email = ? AND user_type = ?',
        (email, user_type)
    )
    user = cursor.fetchone()
    
    if user is None:
        conn.close()
        return jsonify({'success': False, 'message': 'Invalid credentials'}), 401
    
    # Check password
    if bcrypt.checkpw(password.encode('utf-8'), user['password']):
        token = generate_token(user['id'], user['user_type'])
        conn.close()
        
        return jsonify({
            'success': True,
            'user': {
                'id': user['id'],
                'name': user['name'],
                'email': user['email'],
                'user_type': user['user_type']
            },
            'token': token
        }), 200
    else:
        conn.close()
        return jsonify({'success': False, 'message': 'Invalid credentials'}), 401

# Ulema routes
@app.route('/ulemas', methods=['GET'])
def get_ulemas():
    user = get_user_from_token(request)
    if not user:
        return jsonify({'success': False, 'message': 'Unauthorized'}), 401
    
    conn = get_db_connection()
    cursor = conn.execute('SELECT id, name, email, user_type FROM users WHERE user_type = ?', ('ulema',))
    ulemas = cursor.fetchall()
    conn.close()
    
    # Add random online status for demo purposes
    import random
    ulema_list = []
    for ulema in ulemas:
        ulema_list.append({
            'id': ulema['id'],
            'name': ulema['name'],
            'expertise': 'Islamic Inheritance',  # Sample expertise
            'isOnline': random.choice([True, False])
        })
    
    return jsonify({'success': True, 'ulemas': ulema_list}), 200

# Chat routes
@app.route('/chats/start', methods=['POST'])
def start_chat():
    user = get_user_from_token(request)
    if not user:
        return jsonify({'success': False, 'message': 'Unauthorized'}), 401
    
    data = request.json
    ulema_id = data.get('ulema_id')
    inheritance_data = data.get('inheritance_data')
    
    if not ulema_id:
        return jsonify({'success': False, 'message': 'Missing ulema ID'}), 400
    
    conn = get_db_connection()
    
    # Check if chat already exists
    cursor = conn.execute(
        'SELECT id FROM chats WHERE user_id = ? AND ulema_id = ?',
        (user['user_id'], ulema_id)
    )
    existing_chat = cursor.fetchone()
    
    if existing_chat:
        chat_id = existing_chat['id']
    else:
        # Create new chat
        chat_id = str(uuid.uuid4())
        conn.execute(
            'INSERT INTO chats (id, user_id, ulema_id, inheritance_data) VALUES (?, ?, ?, ?)',
            (chat_id, user['user_id'], ulema_id, str(inheritance_data))
        )
        
        # Add initial system message with inheritance data
        system_message = "Inheritance calculation has been shared for verification."
        message_id = str(uuid.uuid4())
        conn.execute(
            'INSERT INTO messages (id, chat_id, sender_id, sender_type, content) VALUES (?, ?, ?, ?, ?)',
            (message_id, chat_id, user['user_id'], 'system', system_message)
        )
    
    conn.commit()
    conn.close()
    
    return jsonify({'success': True, 'chat_id': chat_id}), 200

@app.route('/chats/<chat_id>', methods=['GET'])
def get_chat_details(chat_id):
    user = get_user_from_token(request)
    if not user:
        return jsonify({'success': False, 'message': 'Unauthorized'}), 401
    
    conn = get_db_connection()
    
    # Get chat details
    cursor = conn.execute('''
        SELECT c.id, c.user_id, c.ulema_id, c.created_at, 
               u1.name as user_name, u2.name as ulema_name
        FROM chats c
        JOIN users u1 ON c.user_id = u1.id
        JOIN users u2 ON c.ulema_id = u2.id
        WHERE c.id = ?
    ''', (chat_id,))
    
    chat = cursor.fetchone()
    conn.close()
    
    if not chat:
        return jsonify({'success': False, 'message': 'Chat not found'}), 404
    
    # Check if user is part of this chat
    if user['user_id'] != chat['user_id'] and user['user_id'] != chat['ulema_id']:
        return jsonify({'success': False, 'message': 'Unauthorized access to chat'}), 403
    
    return jsonify({
        'success': True,
        'chat': {
            'id': chat['id'],
            'user_name': chat['user_name'],
            'ulema_name': chat['ulema_name'],
            'created_at': chat['created_at']
        }
    }), 200

@app.route('/chats/<chat_id>/messages', methods=['GET'])
def get_chat_messages(chat_id):
    user = get_user_from_token(request)
    if not user:
        return jsonify({'success': False, 'message': 'Unauthorized'}), 401
    
    conn = get_db_connection()
    
    # Check if user is part of this chat
    cursor = conn.execute(
        'SELECT user_id, ulema_id FROM chats WHERE id = ?',
        (chat_id,)
    )
    chat = cursor.fetchone()
    
    if not chat:
        conn.close()
        return jsonify({'success': False, 'message': 'Chat not found'}), 404
    
    if user['user_id'] != chat['user_id'] and user['user_id'] != chat['ulema_id']:
        conn.close()
        return jsonify({'success': False, 'message': 'Unauthorized access to chat'}), 403
    
    # Get messages
    cursor = conn.execute(
        'SELECT id, sender_id, sender_type, content, timestamp FROM messages WHERE chat_id = ? ORDER BY timestamp',
        (chat_id,)
    )
    messages = cursor.fetchall()
    
    # Mark messages as read if user is the recipient
    if user['user_type'] == 'user':
        conn.execute(
            'UPDATE messages SET is_read = 1 WHERE chat_id = ? AND sender_type = ?',
            (chat_id, 'ulema')
        )
    else:
        conn.execute(
            'UPDATE messages SET is_read = 1 WHERE chat_id = ? AND sender_type = ?',
            (chat_id, 'user')
        )
    
    conn.commit()
    conn.close()
    
    message_list = []
    for message in messages:
        message_list.append({
            'id': message['id'],
            'sender_id': message['sender_id'],
            'sender_type': message['sender_type'],
            'content': message['content'],
            'timestamp': message['timestamp']
        })
    
    return jsonify({'success': True, 'messages': message_list}), 200

@app.route('/chats/<chat_id>/messages', methods=['POST'])
def send_message(chat_id):
    user = get_user_from_token(request)
    if not user:
        return jsonify({'success': False, 'message': 'Unauthorized'}), 401
    
    data = request.json
    content = data.get('content')
    
    if not content:
        return jsonify({'success': False, 'message': 'Message content is required'}), 400
    
    conn = get_db_connection()
    
    # Check if user is part of this chat
    cursor = conn.execute(
        'SELECT user_id, ulema_id FROM chats WHERE id = ?',
        (chat_id,)
    )
    chat = cursor.fetchone()
    
    if not chat:
        conn.close()
        return jsonify({'success': False, 'message': 'Chat not found'}), 404
    
    if user['user_id'] != chat['user_id'] and user['user_id'] != chat['ulema_id']:
        conn.close()
        return jsonify({'success': False, 'message': 'Unauthorized access to chat'}), 403
    
    # Determine sender type
    sender_type = 'user' if user['user_id'] == chat['user_id'] else 'ulema'
    
    # Create message
    message_id = str(uuid.uuid4())
    conn.execute(
        'INSERT INTO messages (id, chat_id, sender_id, sender_type, content) VALUES (?, ?, ?, ?, ?)',
        (message_id, chat_id, user['user_id'], sender_type, content)
    )
    
    conn.commit()
    conn.close()
    
    return jsonify({'success': True, 'message': 'Message sent successfully'}), 201

@app.route('/ulema/chats', methods=['GET'])
def get_ulema_chats():
    user = get_user_from_token(request)
    if not user or user['user_type'] != 'ulema':
        return jsonify({'success': False, 'message': 'Unauthorized'}), 401
    
    conn = get_db_connection()
    
    # Get all chats for this ulema
    cursor = conn.execute('''
        SELECT c.id, c.user_id, u.name as user_name, c.created_at
        FROM chats c
        JOIN users u ON c.user_id = u.id
        WHERE c.ulema_id = ?
        ORDER BY c.created_at DESC
    ''', (user['user_id'],))
    
    chats = cursor.fetchall()
    chat_list = []
    
    for chat in chats:
        # Get last message
        cursor = conn.execute(
            'SELECT content, timestamp FROM messages WHERE chat_id = ? ORDER BY timestamp DESC LIMIT 1',
            (chat['id'],)
        )
        last_message = cursor.fetchone()
        
        # Count unread messages
        cursor = conn.execute(
            'SELECT COUNT(*) as count FROM messages WHERE chat_id = ? AND sender_type = ? AND is_read = 0',
            (chat['id'], 'user')
        )
        unread = cursor.fetchone()
        
        chat_list.append({
            'id': chat['id'],
            'user_name': chat['user_name'],
            'last_message': last_message['content'] if last_message else '',
            'last_message_time': last_message['timestamp'] if last_message else chat['created_at'],
            'unread_count': unread['count']
        })
    
    conn.close()
    
    return jsonify({'success': True, 'chats': chat_list}), 200

if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=6000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.0.108:5000
Press CTRL+C to quit
127.0.0.1 - - [15/May/2025 20:33:38] "OPTIONS /auth/signup HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2025 20:33:39] "POST /auth/signup HTTP/1.1" 201 -
127.0.0.1 - - [15/May/2025 20:34:00] "OPTIONS /auth/login HTTP/1.1" 200 -
C:\Users\ehabq\AppData\Local\Temp\ipykernel_6368\3445739620.py:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'exp': datetime.datetime.utcnow() + datetime.timedelta(days=1)
127.0.0.1 - - [15/May/2025 20:34:01] "POST /auth/login HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2025 20:37:50] "OPTIONS /ulemas HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2025 20:37:50] "GET /ulemas HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2025 20:38:35] "OPTIONS /chats/start HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2025 